In [1]:
import xarray as xr
import numpy as np
from pathlib import Path

# ---------------------------------------------------------
# Paths
# ---------------------------------------------------------

files = {
    "SST": r"C:\OceanF\data\processed\SST\SST_processed.nc",
    "SSS": r"C:\OceanF\data\processed\SSS\SSS_processed.nc",
    "SSA": r"C:\OceanF\data\processed\SSA\SSA_processed.nc",
    "Currents": r"C:\OceanF\data\processed\Currents\Currents_processed.nc",
    "Winds": r"C:\OceanF\data\processed\Winds\Winds_processed.nc",
    "SubsurfaceTemp": r"C:\OceanF\data\processed\SubsurfaceTemp\SubsurfaceTemp_processed.nc",
}

datasets = {}

# ---------------------------------------------------------
# Open datasets
# ---------------------------------------------------------

for name, path in files.items():
    print(f"Opening {name}...")
    
    path = Path(path)
    
    if not path.exists():
        raise FileNotFoundError(f"{name} file not found: {path}")
    
    ds = xr.open_dataset(path)
    
    # Standardize time coordinate name
    if "valid_time" in ds.coords and "time" not in ds.coords:
        ds = ds.rename({"valid_time": "time"})
    
    if "time" not in ds.coords:
        raise ValueError(f"{name} has no usable time coordinate.")
    
    datasets[name] = ds

print("\nAll datasets opened successfully.")

Opening SST...
Opening SSS...
Opening SSA...
Opening Currents...
Opening Winds...
Opening SubsurfaceTemp...

All datasets opened successfully.


In [2]:
# ---------------------------------------------------------
# Find common time period
# ---------------------------------------------------------

common_time = None

for name, ds in datasets.items():
    current_time = ds["time"].values
    
    print(
        f"{name:15s} : "
        f"{current_time[0]} → {current_time[-1]} "
        f"({len(current_time)} days)"
    )
    
    if common_time is None:
        common_time = current_time
    else:
        common_time = np.intersect1d(common_time, current_time)

print("\n" + "=" * 70)
print("COMMON TIME PERIOD")
print("=" * 70)

print("Start :", common_time[0])
print("End   :", common_time[-1])
print("Count :", len(common_time))

SST             : 2025-01-01T00:00:00.000000000 → 2026-03-31T00:00:00.000000000 (455 days)
SSS             : 2025-01-01T00:00:00.000000000 → 2026-08-27T00:00:00.000000000 (604 days)
SSA             : 2025-01-01T00:00:00.000000000 → 2026-01-16T00:00:00.000000000 (381 days)
Currents        : 2025-01-01T00:00:00.000000000 → 2026-09-01T00:00:00.000000000 (609 days)
Winds           : 2025-07-01T00:00:00.000000000 → 2025-12-31T00:00:00.000000000 (184 days)
SubsurfaceTemp  : 2025-07-01T00:00:00.000000000 → 2025-12-31T00:00:00.000000000 (184 days)

COMMON TIME PERIOD
Start : 2025-07-01T00:00:00.000000000
End   : 2025-12-31T00:00:00.000000000
Count : 184


In [3]:
# ---------------------------------------------------------
# Verify daily continuity
# ---------------------------------------------------------

time_diff = np.diff(common_time)

expected_step = np.timedelta64(1, "D")

missing_gaps = np.where(time_diff != expected_step)[0]

print("Number of common dates:", len(common_time))
print("Expected daily step   :", expected_step)
print("Number of gaps        :", len(missing_gaps))

if len(missing_gaps) == 0:
    print("✓ Common timeline is continuous with no gaps.")
else:
    print("✗ Gaps detected:")
    
    for idx in missing_gaps:
        print(
            common_time[idx],
            "→",
            common_time[idx + 1],
            "gap:",
            time_diff[idx]
        )

Number of common dates: 184
Expected daily step   : 1 days
Number of gaps        : 0
✓ Common timeline is continuous with no gaps.


In [4]:
# ---------------------------------------------------------
# Crop every dataset to common timeline
# ---------------------------------------------------------

harmonized = {}

for name, ds in datasets.items():
    
    ds_h = ds.sel(time=common_time)
    
    harmonized[name] = ds_h
    
    print(
        f"{name:15s} : "
        f"{len(ds_h.time)} time steps | "
        f"{ds_h.time.values[0]} → {ds_h.time.values[-1]}"
    )

SST             : 184 time steps | 2025-07-01T00:00:00.000000000 → 2025-12-31T00:00:00.000000000
SSS             : 184 time steps | 2025-07-01T00:00:00.000000000 → 2025-12-31T00:00:00.000000000
SSA             : 184 time steps | 2025-07-01T00:00:00.000000000 → 2025-12-31T00:00:00.000000000
Currents        : 184 time steps | 2025-07-01T00:00:00.000000000 → 2025-12-31T00:00:00.000000000
Winds           : 184 time steps | 2025-07-01T00:00:00.000000000 → 2025-12-31T00:00:00.000000000
SubsurfaceTemp  : 184 time steps | 2025-07-01T00:00:00.000000000 → 2025-12-31T00:00:00.000000000


In [5]:
# ---------------------------------------------------------
# Verify identical time coordinates
# ---------------------------------------------------------

reference_time = harmonized["SST"]["time"].values

for name, ds in harmonized.items():
    
    same_time = np.array_equal(
        reference_time,
        ds["time"].values
    )
    
    print(f"{name:15s} :", "PASS ✓" if same_time else "FAIL ✗")
    
    if not same_time:
        raise ValueError(
            f"Time coordinate mismatch detected in {name}"
        )

print("\n✓ ALL DATASETS HAVE IDENTICAL TIME COORDINATES")

SST             : PASS ✓
SSS             : PASS ✓
SSA             : PASS ✓
Currents        : PASS ✓
Winds           : PASS ✓
SubsurfaceTemp  : PASS ✓

✓ ALL DATASETS HAVE IDENTICAL TIME COORDINATES


In [9]:
# ============================================================
# COMPACT CROSS-DATASET SPATIAL INVENTORY
# ============================================================

print("=" * 90)
print("CROSS-DATASET SPATIAL GRID INVENTORY")
print("=" * 90)

for name, ds in harmonized.items():

    lat = ds["latitude"].values
    lon = ds["longitude"].values

    lat_diff = np.diff(lat)
    lon_diff = np.diff(lon)

    print(f"\n{name}")
    print("-" * 90)

    print(f"Shape              : {lat.size} × {lon.size}")
    print(f"Latitude range     : {lat[0]:.6f} → {lat[-1]:.6f}")
    print(f"Longitude range    : {lon[0]:.6f} → {lon[-1]:.6f}")

    print(
        f"Latitude spacing   : "
        f"min={lat_diff.min():.8f}, "
        f"max={lat_diff.max():.8f}"
    )

    print(
        f"Longitude spacing  : "
        f"min={lon_diff.min():.8f}, "
        f"max={lon_diff.max():.8f}"
    )

    print(
        f"Latitude ascending : {np.all(lat_diff > 0)}"
    )

    print(
        f"Longitude ascending: {np.all(lon_diff > 0)}"
    )

    print(
        f"Variables          : {list(ds.data_vars)}"
    )

CROSS-DATASET SPATIAL GRID INVENTORY

SST
------------------------------------------------------------------------------------------
Shape              : 500 × 1200
Latitude range     : 5.025000 → 29.975000
Longitude range    : 45.025002 → 104.974998
Latitude spacing   : min=0.04999924, max=0.05000114
Longitude spacing  : min=0.04999542, max=0.05000305
Latitude ascending : True
Longitude ascending: True
Variables          : ['sst']

SSS
------------------------------------------------------------------------------------------
Shape              : 200 × 480
Latitude range     : 5.062500 → 29.937500
Longitude range    : 45.062500 → 104.937500
Latitude spacing   : min=0.12500000, max=0.12500000
Longitude spacing  : min=0.12500000, max=0.12500000
Latitude ascending : True
Longitude ascending: True
Variables          : ['sss']

SSA
------------------------------------------------------------------------------------------
Shape              : 200 × 480
Latitude range     : 5.062500 → 29.9375

In [10]:
# ============================================================
# TEST AGAINST OCEANF TARGET GRID
# ============================================================

TARGET_LAT = np.arange(
    5.0,
    30.0 + 0.25 / 2,
    0.25
)

TARGET_LON = np.arange(
    45.0,
    105.0 + 0.25 / 2,
    0.25
)

print("=" * 90)
print("OCEANF 0.25° TARGET GRID CHECK")
print("=" * 90)

print(f"Target latitude : {TARGET_LAT[0]} → {TARGET_LAT[-1]}")
print(f"Target longitude: {TARGET_LON[0]} → {TARGET_LON[-1]}")
print(f"Target shape    : {len(TARGET_LAT)} × {len(TARGET_LON)}")

print("\n" + "-" * 90)

for name, ds in harmonized.items():

    lat = ds["latitude"].values
    lon = ds["longitude"].values

    lat_same = np.array_equal(lat, TARGET_LAT)
    lon_same = np.array_equal(lon, TARGET_LON)

    print(
        f"{name:15s} | "
        f"Lat: {'PASS ✓' if lat_same else 'NOT ON TARGET'} | "
        f"Lon: {'PASS ✓' if lon_same else 'NOT ON TARGET'}"
    )

OCEANF 0.25° TARGET GRID CHECK
Target latitude : 5.0 → 30.0
Target longitude: 45.0 → 105.0
Target shape    : 101 × 241

------------------------------------------------------------------------------------------
SST             | Lat: NOT ON TARGET | Lon: NOT ON TARGET
SSS             | Lat: NOT ON TARGET | Lon: NOT ON TARGET
SSA             | Lat: NOT ON TARGET | Lon: NOT ON TARGET
Currents        | Lat: NOT ON TARGET | Lon: NOT ON TARGET
Winds           | Lat: NOT ON TARGET | Lon: PASS ✓
SubsurfaceTemp  | Lat: PASS ✓ | Lon: PASS ✓
